# CS6603 Final Project — Group 109
## Step 4: Mitigating Bias (Continues Directly From Step 3)

This notebook starts from the **exact setup already established in Step 3**
and extends it with the Step 4 classifier experiments. Nothing about the
Step 3 definitions is changed here — Step 4 simply adds a trained classifier
on top.

**From Step 3.1 (privileged / unprivileged groups):**
- **Age**: privileged = 15–17, unprivileged = 18–22
- **Sex**: privileged = Male, unprivileged = Female

**From Step 3.2 (outcome variables):**
- **G3**: pass (G3 ≥ 10) = 1, fail = 0
- **Absence** (derived): "Good Attendance" (`absences` ≤ 2, the sample
  median) = 1 (favorable), else 0

**From Step 3.3 (pre-processing mitigation):** Reweighing, computed
**separately for each protected class**, using **G3 as the label** — i.e.
`weight_age` balances Age against G3, and `weight_sex` balances Sex against
G3. These are the same weights already used in Step 3.4; Step 4 reuses them
unchanged (which is why the Absence rows below are not perfectly fair even
after transforming — they inherit G3-based weights, exactly as in Step 3.4).

**Step 4 adds:** for each dependent variable, split into train/test, train
a classifier on the original data, train a second classifier on the
transformed (Reweighed) data using the same per-protected-class weights via
`sample_weight`, then compute SPD/DI on the held-out test set for both
protected classes. All four stages (2 from Step 3 + 2 new from Step 4) are
reported in the FAQ's combined-table format.


## 0. Imports

In [ ]:
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load the Dataset (same as Step 3)

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

def load_student_data(url=DATA_URL, fname="student-por.csv"):
    """Download and load the student performance dataset.
    Falls back to a local copy at ./student-por.csv if offline."""
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            zdata = resp.read()
        with zipfile.ZipFile(io.BytesIO(zdata)) as z:
            with z.open(fname) as f:
                df = pd.read_csv(f, sep=';')
    except Exception as e:
        print(f"Download failed ({e}); trying local file '{fname}'...")
        df = pd.read_csv(fname, sep=';')
    return df

df = load_student_data()
print(df.shape)
df.head()


## 2. Reproduce Step 3's Protected Classes and Outcome Variables

These definitions exactly match Step 3.1 / 3.2 of the report.


In [ ]:
data = df.copy()

# Step 3.1 privileged / unprivileged groups
data['age_privileged'] = data['age'].between(15, 17).astype(int)   # 15-17 privileged, 18-22 unprivileged
data['sex_privileged'] = (data['sex'] == 'M').astype(int)          # Male privileged, Female unprivileged

# Step 3.2 outcome variables
data['label_G3'] = (data['G3'] >= 10).astype(int)                  # 1 = pass
data['label_ABS'] = (data['absences'] <= 2).astype(int)            # 1 = good attendance (favorable)

print("G3 pass rate:          ", round(data['label_G3'].mean(), 4))
print("Good attendance rate:  ", round(data['label_ABS'].mean(), 4))


## 3. Fairness Metrics (same two from Step 3.2)

- **Statistical Parity Difference (SPD)** = P(Y=1 | unprivileged) − P(Y=1 | privileged)
- **Disparate Impact (DI)** = P(Y=1 | unprivileged) / P(Y=1 | privileged)

`rates()` supports an optional `sample_weight`, so the same function
reproduces both the raw Step 3.2 numbers and the weighted Step 3.4 numbers.


In [ ]:
def rates(y, mask, w=None):
    y = np.asarray(y)
    mask = np.asarray(mask).astype(bool)
    if w is None:
        priv_rate = y[mask].mean()
        unpriv_rate = y[~mask].mean()
    else:
        w = np.asarray(w)
        priv_rate = np.average(y[mask], weights=w[mask])
        unpriv_rate = np.average(y[~mask], weights=w[~mask])
    spd = unpriv_rate - priv_rate
    di = (unpriv_rate / priv_rate) if priv_rate != 0 else np.nan
    return spd, di

# Sanity check against Step 3.2 (should print Age: -0.0775/0.9107, Sex: 0.0574/1.0707 for G3)
print("Age -> G3:", rates(data['label_G3'], data['age_privileged']))
print("Sex -> G3:", rates(data['label_G3'], data['sex_privileged']))
print("Age -> Absence:", rates(data['label_ABS'], data['age_privileged']))
print("Sex -> Absence:", rates(data['label_ABS'], data['sex_privileged']))


## 4. Reweighing Weights (same as Step 3.3 / 3.4)

Computed **per protected class**, using **G3 as the label** — matching the
report's Step 3.3 note: *"We reweighted the dataset using G3 as a function
of our dependent variables on both protected classes."* This produces two
weight columns, `weight_age` and `weight_sex`, which are reused unchanged
in Step 4 below.


In [ ]:
def reweighing_weights(protected_attr, label):
    """Return per-instance weights that balance protected_attr against label."""
    protected_attr = np.asarray(protected_attr)
    label = np.asarray(label)
    n = len(label)
    weights = np.ones(n, dtype=float)
    for a in np.unique(protected_attr):
        p_a = (protected_attr == a).mean()
        for y in np.unique(label):
            p_y = (label == y).mean()
            mask = (protected_attr == a) & (label == y)
            p_ay = mask.mean()
            if p_ay > 0:
                weights[mask] = (p_a * p_y) / p_ay
    return weights

data['weight_age'] = reweighing_weights(data['age_privileged'], data['label_G3'])
data['weight_sex'] = reweighing_weights(data['sex_privileged'], data['label_G3'])

# Sanity check against Step 3.4 (should print Age: 0./1., Sex: 0./1. for G3;
# Age: -0.121/0.7971, Sex: 0.0409/1.0758 for Absence)
print("Age -> G3 (weighted):", rates(data['label_G3'], data['age_privileged'], w=data['weight_age']))
print("Sex -> G3 (weighted):", rates(data['label_G3'], data['sex_privileged'], w=data['weight_sex']))
print("Age -> Absence (weighted):", rates(data['label_ABS'], data['age_privileged'], w=data['weight_age']))
print("Sex -> Absence (weighted):", rates(data['label_ABS'], data['sex_privileged'], w=data['weight_sex']))


## 5. Build the Feature Matrix

Features: all columns except the grade columns (`G1`, `G2`, `G3`) and
`absences` (both used as dependent variables). Categorical columns are
one-hot encoded.


In [ ]:
drop_cols = ['G1', 'G2', 'G3', 'absences',
             'age_privileged', 'sex_privileged',
             'label_G3', 'label_ABS', 'weight_age', 'weight_sex']
feature_df = data.drop(columns=drop_cols)

categorical_cols = [c for c in [
    'school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
    'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
] if c in feature_df.columns]
X = pd.get_dummies(feature_df, columns=categorical_cols, drop_first=True)

age_privileged = data['age_privileged'].values.astype(bool)
sex_privileged = data['sex_privileged'].values.astype(bool)

print(X.shape)


## 6. Step 4 Pipeline

For **each dependent variable** (G3, Absence):

1. Split into train/test (70/30, stratified on that label).
2. Train a classifier on the **original** training data.
3. For **each protected class** (Age, Sex), train a second classifier on the
   **same training data**, but weighted by that protected class's
   `weight_age` / `weight_sex` column (via `sample_weight`) — i.e., the
   Step 3 transformed dataset, reused for the classifier stage.
4. Compute SPD/DI on the held-out test set for both stages, for both
   protected classes.

This keeps every protected-class result tied to its own Step 3 weighting,
exactly as the report's Step 3.3/3.4 already established.


In [ ]:
DEP_VARS = {
    "G3": {"label_col": "label_G3", "display": "G3 Pass/Fail"},
    "ABS": {"label_col": "label_ABS", "display": "Good Attendance (absences ≤ 2)"},
}
PROTECTED = {
    "Age": {"mask": age_privileged, "weight_col": "weight_age"},
    "Sex": {"mask": sex_privileged, "weight_col": "weight_sex"},
}

results = {pc: {dep: {} for dep in DEP_VARS} for pc in PROTECTED}
accuracies = {}

for dep_key, dep_info in DEP_VARS.items():
    y = data[dep_info["label_col"]].values

    idx = np.arange(len(X))
    idx_train, idx_test = train_test_split(idx, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
    X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
    y_train, y_test = y[idx_train], y[idx_test]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Classifier trained on the ORIGINAL (unweighted) training data — shared across protected classes
    clf_original = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    clf_original.fit(X_train_scaled, y_train)
    y_pred_original = clf_original.predict(X_test_scaled)
    acc_original = (y_pred_original == y_test).mean()
    accuracies[dep_key] = {"original": acc_original, "transformed": {}}
    print(f"[{dep_key}] original-classifier accuracy: {acc_original:.3f}")

    for pc_name, pc_info in PROTECTED.items():
        mask_full = pc_info["mask"]
        w_all = data[pc_info["weight_col"]].values
        w_train = w_all[idx_train]

        # Classifier trained on the TRANSFORMED data for THIS protected class's weighting
        clf_transformed = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
        clf_transformed.fit(X_train_scaled, y_train, sample_weight=w_train)
        y_pred_transformed = clf_transformed.predict(X_test_scaled)
        acc_transformed = (y_pred_transformed == y_test).mean()
        accuracies[dep_key]["transformed"][pc_name] = acc_transformed

        mask_test = mask_full[idx_test]
        row1 = rates(y, mask_full)                          # Original Dataset (Step 3.2)
        row2 = rates(y, mask_full, w=w_all)                  # After Transforming Dataset (Step 3.4)
        row3 = rates(y_pred_original, mask_test)             # After Training Classifier on Original (Step 4)
        row4 = rates(y_pred_transformed, mask_test)          # After Training Classifier on Transformed (Step 4)

        results[pc_name][dep_key] = {
            "SPD": [row1[0], row2[0], row3[0], row4[0]],
            "DI":  [row1[1], row2[1], row3[1], row4[1]],
        }

print("\nDone.")


## 7. FAQ-Format Results Tables

Each table follows the FAQ's example structure: 4 rows (stages), with
"Change Compared to Previous" comparing each row to the one immediately
above it. Rows 1–2 reproduce Step 3.2/3.4 exactly; rows 3–4 are new in
Step 4.


In [ ]:
STAGES = [
    "Original Dataset",
    "After Transforming Dataset",
    "After Training Classifier on Original Dataset",
    "After Training Classifier on Transformed Dataset",
]

def fair_distance(value, metric):
    return abs(value) if metric == "SPD" else abs(value - 1)

def change_label(old_value, new_value, metric):
    old_dist = fair_distance(old_value, metric)
    new_dist = fair_distance(new_value, metric)
    diff = new_dist - old_dist
    if abs(diff) < 1e-9:
        return "No change"
    if abs(diff) < 0.01:
        return "No change / very minimal positive change" if diff < 0 else "No change / very minimal negative change"
    return "Positive change" if diff < 0 else "Negative change"

def build_faq_table(pc_name, dep_key, metric):
    vals = results[pc_name][dep_key][metric]
    rows = []
    for i, stage in enumerate(STAGES):
        change = "NA" if i == 0 else change_label(vals[i - 1], vals[i], metric)
        rows.append({"Stage": stage, metric: round(vals[i], 5), "Change Compared to Previous": change})
    return pd.DataFrame(rows)

# Example: Age -> G3, Disparate Impact
build_faq_table("Age", "G3", "DI")


### 7.1 Age → G3 Pass/Fail

In [ ]:
for metric in ["DI", "SPD"]:
    t = build_faq_table("Age", "G3", metric)
    print(f"\n-- Age -> G3: {metric} --")
    print(t.to_string(index=False))


### 7.2 Age → Good Attendance

In [ ]:
for metric in ["DI", "SPD"]:
    t = build_faq_table("Age", "ABS", metric)
    print(f"\n-- Age -> Attendance: {metric} --")
    print(t.to_string(index=False))


### 7.3 Sex → G3 Pass/Fail

In [ ]:
for metric in ["DI", "SPD"]:
    t = build_faq_table("Sex", "G3", metric)
    print(f"\n-- Sex -> G3: {metric} --")
    print(t.to_string(index=False))


### 7.4 Sex → Good Attendance

In [ ]:
for metric in ["DI", "SPD"]:
    t = build_faq_table("Sex", "ABS", metric)
    print(f"\n-- Sex -> Attendance: {metric} --")
    print(t.to_string(index=False))


## 8. Visualization: 4-Stage Progression Charts

One bar chart per protected class × dependent variable × metric
combination (8 charts total).


In [ ]:
STAGE_LABELS_SHORT = ["Original\nDataset", "After\nTransforming", "Classifier on\nOriginal", "Classifier on\nTransformed"]
DEP_DISPLAY = {"G3": "G3 Pass/Fail", "ABS": "Good Attendance"}
BAR_COLORS = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

figures = {}
for pc_name in PROTECTED:
    for dep_key in DEP_VARS:
        for metric in ["SPD", "DI"]:
            vals = results[pc_name][dep_key][metric]
            fig, ax = plt.subplots(figsize=(7, 4.2))
            bars = ax.bar(STAGE_LABELS_SHORT, vals, color=BAR_COLORS)
            fair_value = 0 if metric == "SPD" else 1
            ax.axhline(fair_value, color="gray", linestyle="--", linewidth=1, label=f"Fair value ({fair_value})")
            for b, v in zip(bars, vals):
                ax.text(b.get_x() + b.get_width() / 2, v + (0.01 if v >= 0 else -0.03), f"{v:.4f}",
                        ha="center", va="bottom" if v >= 0 else "top", fontsize=9)
            metric_name = "Statistical Parity Difference" if metric == "SPD" else "Disparate Impact"
            ax.set_title(f"{metric_name} — {pc_name} → {DEP_DISPLAY[dep_key]}")
            ax.set_ylabel(metric_name)
            ax.legend(loc="best", fontsize=8)
            plt.tight_layout()
            fname = f"step4_{pc_name}_{dep_key}_{metric}.png"
            plt.savefig(fname, dpi=150)
            figures[(pc_name, dep_key, metric)] = fname
            plt.show()

print("Saved charts:", list(figures.values()))


## 9. Interpretation Notes

- Rows 1–2 of every table reproduce Step 3.2 / Step 3.4 exactly, confirming
  Step 4 is built directly on top of the completed Step 3 work.
- The classifier stage (rows 3–4) shows whether Reweighing continues to
  help once a real predictive model is introduced. Because the transformed
  classifier for each dependent variable is trained per protected class
  (using that class's own G3-based weights), the Age and Sex columns may
  show different classifier accuracies for the same dependent variable —
  this is expected and reflects each protected class's own weighting.
- Since Absence's Reweighing weights are inherited from G3 (per Step 3.3),
  the Absence rows are not expected to reach perfect fairness at any stage
  — this mirrors the pattern already visible in Step 3.4.
- Copy the tables from Section 7 and the charts from Section 8 directly
  into the final report (8 tables, 8 figures: 2 protected classes × 2
  dependent variables × 2 metrics).
